# Initialization

In [1]:
%pip install implicit

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn.preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from surprise import Dataset, Reader
from surprise import NormalPredictor, SVD, accuracy
from implicit.als import AlternatingLeastSquares
import uuid
import gc
import os
import psutil


/workspaces/mle-recsys-start/env_recsys_py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [4]:
def show_memory():
    process = psutil.Process(os.getpid())

    ram_gb = process.memory_info().rss / 1024**3

    print(f"Память процесса Python: {ram_gb:.2f} GB")

In [5]:
items = pd.read_parquet("/workspaces/mle-recsys-start/items.par")
events = pd.read_parquet("/workspaces/mle-recsys-start/events.par")

In [6]:
display(items.head())

,book_id,author,title,description,genre_and_votes,num_pages,average_rating,ratings_count,text_reviews_count,publisher,...,genre_and_votes_dict.Young Adult-Young Adult Fantasy,genre_and_votes_dict.Young Adult-Young Adult Historical Fiction,genre_and_votes_dict.Young Adult-Young Adult Paranormal,genre_and_votes_dict.Young Adult-Young Adult Romance,genre_and_votes_dict.Young Adult-Young Adult Science Fiction,genre_and_votes_dict.k,genre_and_votes_dict.nd Grade,genre_and_votes_dict.st Century,genre_and_votes_dict.st Grade,genre_and_votes_dict.th Century
3.0,6066819,Jennifer Weiner,Best Friends Forever,Addie Downs and Valerie Adler were eight when ...,"{'Womens Fiction-Chick Lit': 739, 'Fiction': 442}",368,3.49,51184,3282,Atria Books,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6.0,378460,Michael Halberstam,The Wanting of Levine,,"{'Politics': 1, 'Humor': 1}",<NA>,4.38,12,4,Berkley Publishing Group,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15.0,89375,"Don Piper, Cecil Murphey",90 Minutes in Heaven: A True Story of Death an...,As he is driving home from a minister's confer...,"{'Christian': 395, 'Nonfiction': 392, 'Religio...",<NA>,3.91,68157,2885,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16.0,89376,Randy Alcorn,Heaven,What is Heaven really going to be like? What w...,"{'Christian': 225, 'Religion-Theology': 154, '...",533,4.26,7345,566,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17.0,89377,Jennifer L. Holm,Penny from Heaven,It's 1953 and 11-year-old Penny dreams of a su...,"{'Historical-Historical Fiction': 284, 'Childr...",288,3.98,6949,615,Random House Books for Young Readers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
display(events.head())

,user_id,book_id,started_at,read_at,is_read,rating,is_reviewed,started_at_month
0,8842281e1d1347389f2ab93d60773d4d,22034,2015-07-12,2015-07-17,True,5,False,1435708800000000000
1,8842281e1d1347389f2ab93d60773d4d,22318578,2015-06-07,2015-08-09,True,5,True,1433116800000000000
2,8842281e1d1347389f2ab93d60773d4d,22551730,2015-06-24,2015-07-11,True,4,True,1433116800000000000
3,8842281e1d1347389f2ab93d60773d4d,22816087,2015-09-27,2015-11-04,True,5,True,1441065600000000000
4,8842281e1d1347389f2ab93d60773d4d,20527133,2015-05-05,2015-09-06,True,0,False,1430438400000000000


# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [8]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01")

train_test_global_time_split_idx = (
    events["started_at"] < train_test_global_time_split_date
)

events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# количество пользователей, которые есть и в train, и в test
common_users = users_train[users_train.isin(users_test)]

print(len(users_train), len(users_test), len(common_users))

536293 138148 131924


In [9]:
cold_users = users_test[~users_test.isin(users_train)]

print(len(cold_users))

6224


In [10]:
top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["book_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# Сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(
    by="popularity_score",
    ascending=False
)

# Выбираем топ-100 книг с рейтингом не ниже 4
top_k_pop_items = item_popularity[
    item_popularity["avg_rating"] >= 4
].head(100)

In [11]:
# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("book_id")[["author", "title", "genre_and_votes", "publication_year"]], on="book_id")

with pd.option_context('display.max_rows', 100):
    display(top_k_pop_items[["book_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,book_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,21647,4.069663,0.397052,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,17819,4.065043,0.326464,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,16634,4.331249,0.324709,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,14571,4.163681,0.273431,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
4,17927395,Sarah J. Maas,A Court of Mist and Fury (A Court of Thorns an...,2016,13026,4.443191,0.260846,"{'Fantasy': 10186, 'Romance': 3346, 'Young Adu..."
5,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,2004,13231,4.303907,0.256646,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict..."
6,21853621,Kristin Hannah,The Nightingale,2015,13011,4.313197,0.252923,"{'Historical-Historical Fiction': 10105, 'Fict..."
7,13206900,Marissa Meyer,"Winter (The Lunar Chronicles, #4)",2015,13230,4.220937,0.251679,"{'Fantasy': 4835, 'Young Adult': 4672, 'Scienc..."
8,9969571,Ernest Cline,Ready Player One,2011,12679,4.041801,0.230960,"{'Science Fiction': 10560, 'Fiction': 7713, 'Y..."
9,6,"J.K. Rowling, Mary GrandPré",Harry Potter and the Goblet of Fire (Harry Pot...,2002,11953,4.274408,0.230265,"{'Fantasy': 48257, 'Young Adult': 15483, 'Fict..."


In [12]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(
        top_k_pop_items[["book_id", "avg_rating"]],
        on="book_id",
        how="left"
    )

cold_user_items_no_avg_rating_idx = (
    cold_users_events_with_recs["avg_rating"].isnull()
)

cold_user_recs = cold_users_events_with_recs[
    ~cold_user_items_no_avg_rating_idx
][["user_id", "book_id", "rating", "avg_rating"]]

In [13]:
# Рассчитываем RMSE
rmse = np.sqrt(
    mean_squared_error(
        cold_user_recs["rating"],
        cold_user_recs["avg_rating"]
    )
)

# Рассчитываем MAE
mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")


RMSE: 1.65
MAE: 1.12


In [14]:
# посчитаем покрытие холодных пользователей рекомендациями
cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.83
Среднее покрытие пользователей: 0.57


# === Знакомство: первые персональные рекомендации

In [15]:
# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'book_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [16]:
surprise_test_set = list(events_test[['user_id', 'book_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

In [17]:
rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)
                     
print(rmse, mae)

RMSE: 1.3469
MAE:  0.9899
1.3468989130561508 0.9898677202113105


In [18]:
random_model = NormalPredictor()
random_model.fit(surprise_train_set)

random_predictions = random_model.test(surprise_test_set)
random_mae = accuracy.mae(random_predictions, verbose=False)

# Используем предсказания ранее обученной SVD
svd_mae = accuracy.mae(svd_predictions, verbose=False)

mae_difference_percent = (
    (random_mae - svd_mae) / svd_mae * 100
)

print(f"MAE NormalPredictor: {random_mae:.4f}")
print(f"MAE SVD: {svd_mae:.4f}")
print(f"Разница: {mae_difference_percent:.2f}%")

MAE NormalPredictor: 1.4960
MAE SVD: 0.9899
Разница: 51.13%


Факультативное задание

Задание. Удалите из events события для редких айтемов — таких, с которыми взаимодействовало менее N пользователей. Возьмите небольшое N, например 2–3 пользователя. Получите рекомендации, посчитайте метрики, оцените, как они изменились. Подумайте, с чем могут быть связаны такие изменения.

In [19]:
N = 3

item_user_counts = (
    events_train
    .groupby("book_id", sort=False)["user_id"]
    .nunique()
)

popular_items = item_user_counts.index[
    item_user_counts >= N
]

events_train_filtered = events_train.loc[
    events_train["book_id"].isin(popular_items)
].copy()

print(f"Событий до: {len(events_train):,}")
print(f"Событий после: {len(events_train_filtered):,}")
print(f"Удалено событий: {len(events_train) - len(events_train_filtered):,}")
print(f"Удалено книг: {(item_user_counts < N).sum():,}")

del item_user_counts, popular_items
gc.collect()

Событий до: 12,406,925
Событий после: 12,402,318
Удалено событий: 4,607
Удалено книг: 3,176


0

In [20]:
# Используем только необходимые столбцы
item_stats = (
    events_train_filtered[
        ["book_id", "rating", "user_id"]
    ]
    .groupby("book_id", sort=False)
    .agg(
        avg_rating=("rating", "mean"),
        user_count=("user_id", "nunique")
    )
)

# Выбираем топ-100 популярных книг
top_k_pop_items_filtered = (
    item_stats
    .sort_values(
        by=["user_count", "avg_rating"],
        ascending=[False, False]
    )
    .head(100)
    .reset_index()
)

display(top_k_pop_items_filtered.head())

del item_stats
gc.collect()

,book_id,avg_rating,user_count
0,22557272,3.522489,44443
1,11870085,4.074039,43855
2,11735983,3.657509,32506
3,13335037,3.821637,30494
4,7260188,3.691513,29998


0

In [21]:
# Объединяем тестовые события с рекомендациями
test_small = events_test[
    ["user_id", "book_id", "rating"]
]

events_with_recs_filtered = test_small.merge(
    top_k_pop_items_filtered[["book_id", "avg_rating"]],
    on="book_id",
    how="left"
)

# Оставляем события, для которых есть прогноз
recs_filtered = events_with_recs_filtered.dropna(
    subset=["rating", "avg_rating"]
).copy()

# Рассчитываем метрики
rmse_filtered = np.sqrt(
    mean_squared_error(
        recs_filtered["rating"],
        recs_filtered["avg_rating"]
    )
)

mae_filtered = mean_absolute_error(
    recs_filtered["rating"],
    recs_filtered["avg_rating"]
)

print(f"RMSE после фильтрации: {rmse_filtered:.4f}")
print(f"MAE после фильтрации: {mae_filtered:.4f}")

RMSE после фильтрации: 1.5221
MAE после фильтрации: 1.0749


In [22]:
hit_ratio_filtered = (
    events_with_recs_filtered["avg_rating"]
    .notna()
    .groupby(events_with_recs_filtered["user_id"])
    .mean()
    .mean()
)
print(f"Hit Ratio: {hit_ratio_filtered:.4f}")

Hit Ratio: 0.1044


In [23]:
# Исходные рекомендации
events_with_recs_before = events_test.merge(
    top_k_pop_items[["book_id", "avg_rating"]],
    on="book_id",
    how="left"
)

recs_before = events_with_recs_before.dropna(
    subset=["rating", "avg_rating"]
)

rmse_before = np.sqrt(
    mean_squared_error(
        recs_before["rating"],
        recs_before["avg_rating"]
    )
)

mae_before = mean_absolute_error(
    recs_before["rating"],
    recs_before["avg_rating"]
)

hit_ratio_before = (
    events_with_recs_before
    .groupby("user_id")["avg_rating"]
    .apply(lambda x: x.notna().mean())
    .mean()
)

comparison = pd.DataFrame({
    "Метрика": ["RMSE", "MAE", "Hit Ratio"],
    "До фильтрации": [
        rmse_before,
        mae_before,
        hit_ratio_before
    ],
    "После фильтрации": [
        rmse_filtered,
        mae_filtered,
        hit_ratio_filtered
    ]
})

comparison["Изменение"] = (
    comparison["После фильтрации"]
    - comparison["До фильтрации"]
)

display(comparison.round(4))

,Метрика,До фильтрации,После фильтрации,Изменение
0,RMSE,1.4644,1.5221,0.0577
1,MAE,0.9806,1.0749,0.0943
2,Hit Ratio,0.0952,0.1044,0.0092


Задание

Создайте функцию, которая позволит получить рекомендации для конкретного пользователя, используя описанный подход. Дополните прекод.

In [24]:
import heapq

def get_recommendations_svd(
    user_id,
    all_items,
    events,
    model,
    include_seen=True,
    n=5
):
    seen_items = set()

    if not include_seen:
        seen_items = set(
            events.loc[
                events["user_id"] == user_id,
                "book_id"
            ]
        )

    predictions = (
        model.predict(user_id, book_id)
        for book_id in all_items
        if book_id not in seen_items
    )

    top_predictions = heapq.nlargest(
        n,
        predictions,
        key=lambda x: x.est
    )

    return pd.DataFrame(
        [(p.iid, p.est) for p in top_predictions],
        columns=["book_id", "score"]
    )

In [25]:
recommendations = get_recommendations_svd(
    user_id=123,
    all_items=set(items["book_id"].unique()),
    events=events,
    model=svd_model,
    include_seen=False,
    n=5
)

display(recommendations)

,book_id,score
0,29844341,4.711874
1,54741,4.692017
2,22552026,4.668152
3,22037424,4.580056
4,17874649,4.577286


In [26]:
recommendations = get_recommendations_svd(
    user_id=1296647,
    all_items=set(items["book_id"].unique()),
    events=events,
    model=svd_model,
    include_seen=True,
    n=5
)

# Третья рекомендация по убыванию score
print(recommendations.iloc[2]["book_id"])

22552026.0


In [27]:
# Выбираем случайного пользователя
user_id = events_train["user_id"].sample().iat[0]

print(f"user_id: {user_id}")

# Получаем рекомендации
user_recommendations = get_recommendations_svd(
    user_id=user_id,
    all_items=set(items["book_id"].unique()),
    events=events_train,
    model=svd_model
)

print(user_recommendations.dtypes)
display(user_recommendations.head(10))
print(user_recommendations.columns.tolist())

user_id: d1511e8d488f768d4febcff99c61c881
book_id    int64
score      int64
dtype: object


,book_id,score
0,1,5
1,3,5
2,2,5
3,5,5
4,10,5


['book_id', 'score']


In [28]:
# Выберем случайного пользователя из тренировочной выборки
user_id = events_train["user_id"].sample().iat[0]

print(f"user_id: {user_id}")

# История пользователя
print("История (последние события, recent)")

user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(
        items[["book_id", "author", "title", "genre_and_votes"]],
        on="book_id",
        how="left"
    )
)

user_history_to_print = user_history[
    ["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]
].tail(10)

display(user_history_to_print)

# Рекомендации
print("Рекомендации")

user_recommendations = get_recommendations_svd(
    user_id,
    set(items["book_id"].unique()),
    events_train,
    svd_model
)

# Исправляем тип book_id
user_recommendations["book_id"] = (
    user_recommendations["book_id"].astype("int64")
)

# Объединяем рекомендации с информацией о книгах
user_recommendations = user_recommendations.merge(
    items[["book_id", "author", "title", "genre_and_votes"]],
    on="book_id",
    how="left"
)

display(user_recommendations)

user_id: 5c4ee1f0cda31adf07dc0a0e7b6d7137
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
70,Karen Traviss,"The Clone Wars (Star Wars: The Clone Wars, #1)",2013-03-14,2013-03-19,3,"{'Media Tie In-Star Wars': 533, 'Science Ficti..."
71,James Luceno,Darth Plagueis,2012-12-07,2013-01-30,5,"{'Media Tie In-Star Wars': 1528, 'Science Fict..."
72,Drew Karpyshyn,"Rule of Two (Star Wars: Darth Bane, #2)",2012-10-04,2012-11-03,4,"{'Media Tie In-Star Wars': 1447, 'Science Fict..."
73,Robert Jordan,"The Shadow Rising (Wheel of Time, #4)",2012-08-16,2012-12-04,3,"{'Fantasy': 8981, 'Fiction': 1011, 'Fantasy-Ep..."
74,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,2017-05-28,2017-09-27,4,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
75,Robert Jordan,"The Great Hunt (Wheel of Time, #2)",2012-05-09,2012-06-16,5,"{'Fantasy': 11050, 'Fiction': 1258, 'Fantasy-E..."
76,Rick Riordan,The Battle of the Labyrinth (Percy Jackson and...,2013-02-02,2013-03-28,4,"{'Fantasy': 12074, 'Young Adult': 5159, 'Fanta..."
77,Christopher Paolini,"Inheritance (The Inheritance Cycle, #4)",2013-06-21,2013-08-14,3,"{'Fantasy': 8118, 'Young Adult': 1885, 'Fantas..."
78,Christopher Paolini,"Brisingr (The Inheritance Cycle, #3)",2012-07-02,2013-05-21,2,"{'Fantasy': 10921, 'Young Adult': 2693, 'Ficti..."
79,Christopher Paolini,"Eldest (The Inheritance Cycle, #2)",2012-01-01,2012-07-01,2,"{'Fantasy': 13121, 'Young Adult': 3442, 'Ficti..."


Рекомендации


,book_id,score,author,title,genre_and_votes
0,21,5,Bill Bryson,A Short History of Nearly Everything,"{'Nonfiction': 7715, 'Science': 5973, 'History..."
1,1554,5,"Sophocles, J.E. Thomas","Oedipus Rex (The Theban Plays, #1)","{'Classics': 4512, 'Plays': 3029, 'Drama': 100..."
2,8852,5,William Shakespeare,Macbeth,"{'Classics': 16116, 'Plays': 8310, 'Fiction': ..."
3,156427,5,Robert Bloch,Psycho,"{'Horror': 1578, 'Fiction': 444, 'Classics': 4..."
4,17332243,5,Nicola Griffith,Hild,"{'Historical-Historical Fiction': 1166, 'Ficti..."


Факультативное задание

Добавьте в events события для нового пользователя, как если бы он прочитал те книги, которые нравятся вам. Получите рекомендации для этого пользователя. Оцените, насколько они релевантны вашим интересам.

In [29]:
# ==========================================
# 1. Создаём нового пользователя
# ==========================================

new_user_id = uuid.uuid4().hex

print(f"ID нового пользователя: {new_user_id}")


# ==========================================
# 2. Выбираем 5 книг из каталога
# ==========================================

favorite_items = (
    events['book_id']
    .dropna()
    .drop_duplicates()
    .head(5)
    .tolist()
)

print(f"Выбранные книги: {favorite_items}")


# ==========================================
# 3. Создаём события нового пользователя
# ==========================================

new_events = pd.DataFrame({
    'user_id': [new_user_id] * len(favorite_items),
    'book_id': favorite_items,
    'rating': [5.0] * len(favorite_items)
})

all_items = events["book_id"].dropna().unique()

print("\nСобытия нового пользователя:")
display(new_events)


# ==========================================
# 4. Подготавливаем данные для SVD
# ==========================================

# Используем только необходимые столбцы
svd_columns = ['user_id', 'book_id', 'rating']

# Убираем события без рейтинга
events_rated = events.loc[
    events['rating'].notna(),
    svd_columns
]

print(f"\nСобытий с рейтингами: {len(events_rated)}")


# ==========================================
# 5. Добавляем нового пользователя
# ==========================================

events_svd = pd.concat(
    [events_rated, new_events[svd_columns]],
    ignore_index=True
)

print(f"Количество событий до: {len(events_rated)}")
print(f"Количество событий после: {len(events_svd)}")


# ==========================================
# 6. Проверяем результат
# ==========================================

new_user_data = events_svd[
    events_svd['user_id'] == new_user_id
]

print("\nСобытия нового пользователя:")
display(new_user_data)

print(
    "\nКоличество книг нового пользователя:",
    len(new_user_data)
)


# ==========================================
# 7. Проверяем расход памяти
# ==========================================

memory_mb = (
    events_svd.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"\nПамять events_svd: {memory_mb:.2f} MB")

gc.collect()

ID нового пользователя: bfe86f1a28a64bbe94ef909b7b15aa4e
Выбранные книги: [22034, 22318578, 22551730, 22816087, 20527133]

События нового пользователя:


,user_id,book_id,rating
0,bfe86f1a28a64bbe94ef909b7b15aa4e,22034,5.0
1,bfe86f1a28a64bbe94ef909b7b15aa4e,22318578,5.0
2,bfe86f1a28a64bbe94ef909b7b15aa4e,22551730,5.0
3,bfe86f1a28a64bbe94ef909b7b15aa4e,22816087,5.0
4,bfe86f1a28a64bbe94ef909b7b15aa4e,20527133,5.0



Событий с рейтингами: 12887523
Количество событий до: 12887523
Количество событий после: 12887528

События нового пользователя:


,user_id,book_id,rating
12887523,bfe86f1a28a64bbe94ef909b7b15aa4e,22034,5.0
12887524,bfe86f1a28a64bbe94ef909b7b15aa4e,22318578,5.0
12887525,bfe86f1a28a64bbe94ef909b7b15aa4e,22551730,5.0
12887526,bfe86f1a28a64bbe94ef909b7b15aa4e,22816087,5.0
12887527,bfe86f1a28a64bbe94ef909b7b15aa4e,20527133,5.0



Количество книг нового пользователя: 5

Память events_svd: 1192.18 MB


0

In [30]:
# Книги, которые пользователь уже оценил
seen_items = set(new_events['book_id'])

# Все доступные книги
all_items = events_svd['book_id'].unique()

# Предсказываем рейтинги для непрочитанных книг
recommendations = []

for book_id in all_items:
    if book_id not in seen_items:
        prediction = svd_model.predict(
            uid=new_user_id,
            iid=book_id
        )

        recommendations.append({
            'book_id': book_id,
            'score': prediction.est
        })

# Выбираем 10 лучших рекомендаций
recommendations_df = (
    pd.DataFrame(recommendations)
    .sort_values('score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(recommendations_df)

,book_id,score
0,29844341,4.711874
1,54741,4.692017
2,22552026,4.668152
3,22037424,4.580056
4,17874649,4.577286
5,6898978,4.574685
6,24812,4.573624
7,25793186,4.571148
8,31422889,4.556868
9,580213,4.541967


# === Базовые подходы: коллаборативная фильтрация

In [31]:
# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["book_id"])
items["item_id_enc"] = item_encoder.transform(items["book_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["book_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["book_id"])


/tmp/ipykernel_1944/1525149691.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_1944/1525149691.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])
/tmp/ipykernel_1944/1525149691.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

In [32]:
# Вариант 1: Прямо из полученной колонки
print(events_train["item_id_enc"].max())


43304


In [33]:
matrix_size_gb = (
    len(user_encoder.classes_) *
    len(item_encoder.classes_)
) // (1024 ** 3)

print(int(matrix_size_gb))


21


In [34]:
from scipy.sparse import coo_matrix

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

user_item_matrix_train = coo_matrix(
    (
        events_train["rating"].fillna(0).to_numpy(dtype=np.float32),
        (
            events_train["user_id_enc"].to_numpy(dtype=np.int32),
            events_train["item_id_enc"].to_numpy(dtype=np.int32)
        )
    ),
    shape=(n_users, n_items),
    dtype=np.float32
).tocsr()

user_item_matrix_train.eliminate_zeros()

memory_mb = (
    user_item_matrix_train.data.nbytes
    + user_item_matrix_train.indices.nbytes
    + user_item_matrix_train.indptr.nbytes
) / 1024**2

print(f"Размер матрицы: {user_item_matrix_train.shape}")
print(f"Ненулевых элементов: {user_item_matrix_train.nnz:,}")
print(f"Память CSR: {memory_mb:.2f} MB")

Размер матрицы: (542517, 43312)
Ненулевых элементов: 11,407,825
Память CSR: 89.10 MB


In [35]:
als_model = AlternatingLeastSquares(
    factors=50,
    iterations=10,
    regularization=0.05,
    random_state=0,
    num_threads=2,
    use_gpu=False
)

als_model.fit(user_item_matrix_train)

/workspaces/mle-recsys-start/env_recsys_py312/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 10/10 [00:37<00:00,  3.78s/it]


In [36]:
def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])
    
    return recommendations

# === Базовые подходы: контентные рекомендации

In [37]:
items["genre_and_votes"] = items["genre_and_votes"].apply(eval)

In [38]:
def get_genres(items):

    """ 
    извлекает список жанров по всем книгам, 
    подсчитывает долю голосов по каждому их них
    """
    
    genres_counter = {}
    
    for k, v, in items.iterrows():
        genre_and_votes = v["genre_and_votes"]
        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue
        for genre, votes in genre_and_votes.items():
            # увеличиваем счётчик жанров
            try:
                genres_counter[genre] += votes
            except KeyError:
                genres_counter[genre] = 0

    genres = pd.Series(genres_counter, name="votes")
    genres = genres.to_frame()
    genres = genres.reset_index().rename(columns={"index": "name"})
    genres.index.name = "genre_id"
    
    return genres
   
genres = get_genres(items)

In [39]:
genres["score"] = genres["votes"] / genres["votes"].sum()
genres.sort_values(by="score", ascending=False).head(10)

,name,votes,score
genre_id,,,
25,Fantasy,6850060,0.149651
1,Fiction,6406256,0.139955
38,Classics,3414934,0.074605
18,Young Adult,3296951,0.072027
34,Romance,2422614,0.052926
5,Nonfiction,1737406,0.037957
16,Historical-Historical Fiction,1531205,0.033452
20,Mystery,1371196,0.029956
24,Science Fiction,1218917,0.026629


In [40]:
def get_item2genre_matrix(genres, items):

    genre_names_to_id = genres.reset_index().set_index("name")["genre_id"].to_dict()
    
    # list to build CSR matrix
    genres_csr_data = []
    genres_csr_row_idx = []
    genres_csr_col_idx = []
    
    for item_idx, (k, v) in enumerate(items.iterrows()):
        if v["genre_and_votes"] is None:
            continue
        for genre_name, votes in v["genre_and_votes"].items():
            genre_idx = genre_names_to_id[genre_name]
            genres_csr_data.append(int(votes))
            genres_csr_row_idx.append(item_idx)
            genres_csr_col_idx.append(genre_idx)

    genres_csr = scipy.sparse.csr_matrix((genres_csr_data, (genres_csr_row_idx, genres_csr_col_idx)), shape=(len(items), len(genres)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)
    
    return genres_csr

In [41]:
items_train_genres = get_item2genre_matrix(genres, items)
print(items_train_genres)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210895 stored elements and shape (43312, 815)>
  Coords	Values
  (0, 0)	0.6257408975444538
  (0, 1)	0.37425910245554617
  (1, 2)	0.5
  (1, 3)	0.5
  (2, 4)	0.2633333333333333
  (2, 5)	0.2613333333333333
  (2, 6)	0.09466666666666666
  (2, 7)	0.06933333333333333
  (2, 8)	0.064
  (2, 9)	0.062
  (2, 10)	0.058
  (2, 11)	0.048666666666666664
  (2, 12)	0.042666666666666665
  (2, 13)	0.036
  (3, 4)	0.3129346314325452
  (3, 5)	0.13212795549374132
  (3, 6)	0.07510431154381085
  (3, 11)	0.11126564673157163
  (3, 13)	0.07927677329624479
  (3, 14)	0.2141863699582754
  (3, 15)	0.07510431154381085
  (4, 1)	0.1111111111111111
  (4, 16)	0.4930555555555556
  (4, 17)	0.1579861111111111
  (4, 18)	0.13194444444444445
  :	:
  (43307, 19)	0.0391566265060241
  (43307, 38)	0.07981927710843373
  (43307, 68)	0.1355421686746988
  (43307, 78)	0.4427710843373494
  (43307, 79)	0.06325301204819277
  (43308, 100)	0.4263420724094881
  (43308, 101)	0.320224719

In [42]:
items = items.sort_values(by="item_id_enc")
all_items_genres_csr = get_item2genre_matrix(genres, items)

In [43]:
user_id = events_train["user_id"].iloc[0]

user_events = events_train.loc[
    events_train["user_id"] == user_id,
    ["book_id", "rating"]
]

user_items = items[
    items["book_id"].isin(user_events["book_id"])
]

user_items_genres_csr = get_item2genre_matrix(
    genres,
    user_items
)

print("Пользователь:", user_id)
print("Количество книг:", len(user_items))
print("Количество элементов:", user_items_genres_csr.nnz)

Пользователь: 8842281e1d1347389f2ab93d60773d4d
Количество книг: 36
Количество элементов: 238


In [44]:
# вычислим склонность пользователя к жанрам как среднее взвешенное значение популяции на его оценки книг.

# преобразуем пользовательские оценки из списка в вектор-столбец
user_ratings = user_events["rating"].to_numpy() / 5
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_items_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0))

In [45]:
# выведем список жанров, которые предпочитает пользователь

user_genres = genres.copy()
user_genres["score"] = np.ravel(user_genres_scores)
user_genres = user_genres[user_genres["score"] > 0].sort_values(by=["score"], ascending=False)

user_genres.head(5)

,name,votes,score
genre_id,,,
1,Fiction,6406256,0.143925
5,Nonfiction,1737406,0.109859
24,Science Fiction,1218917,0.099539
25,Fantasy,6850060,0.057212
36,History,533811,0.048548


In [46]:
from sklearn.metrics.pairwise import cosine_similarity

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k (по убыванию значений), по сути, индексы книг (encoded)
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k]

In [47]:
selected_items = items[items["item_id_enc"].isin(top_k_indices)]

with pd.option_context("max_colwidth", 100):
   display(selected_items[["author", "title", "genre_and_votes"]]) 

,author,title,genre_and_votes
1383172.0,Neal Stephenson,Cryptonomicon,"{'Fiction': 2319, 'Science Fiction': 1880, 'Historical-Historical Fiction': 820, 'Science Fictio..."
750344.0,David Mitchell,Cloud Atlas,"{'Fiction': 5451, 'Science Fiction': 2390, 'Fantasy': 1654, 'Historical-Historical Fiction': 1102}"
764293.0,"Robert Shea, Robert Anton Wilson",The Illuminatus! Trilogy,"{'Fiction': 457, 'Science Fiction': 311, 'Fantasy': 203, 'Humor': 75, 'Philosophy': 75}"
1520240.0,H.F. Saint,Memoirs of an Invisible Man,"{'Fiction': 78, 'Science Fiction': 53, 'Fantasy': 18, 'Thriller': 9}"
1289189.0,J.G. Ballard,The Unlimited Dream Company,"{'Fiction': 59, 'Science Fiction': 43, 'Fantasy': 22}"


Факультативное задание

Оцените, для какой доли пользователей удалось посчитать метрику NDCG.

In [48]:
# Пользователи, присутствующие в обучающей выборке
train_users = set(events_train["user_id"].unique())

# Пользователи тестовой выборки
test_users = set(events_test["user_id"].unique())

# Пользователи, для которых можно получить
# персонализированные рекомендации на основе обучения
eligible_users = train_users & test_users

total_users = len(test_users)
eligible_count = len(eligible_users)

share = eligible_count / total_users

print(f"Всего пользователей: {total_users:,}")
print(f"Пользователей с возможностью расчёта NDCG: {eligible_count:,}")
print(f"Доля пользователей: {share:.2%}")

Всего пользователей: 138,148
Пользователей с возможностью расчёта NDCG: 131,924
Доля пользователей: 95.49%


In [49]:
def get_similar_books(book_id, model, items, item_encoder, n=5):
    # Преобразуем book_id в закодированный идентификатор
    item_id_enc = int(item_encoder.transform([book_id])[0])

    # Получаем похожие книги
    similar_ids, scores = model.similar_items(
        item_id_enc,
        N=n + 1
    )

    # Создаём DataFrame
    recommendations = pd.DataFrame({
        "item_id_enc": similar_ids,
        "similarity": scores
    })

    # Исключаем исходную книгу
    recommendations = recommendations[
        recommendations["item_id_enc"] != item_id_enc
    ].head(n).copy()

    # Добавляем информацию о книгах
    recommendations = recommendations.merge(
        items[["item_id_enc", "book_id", "author", "title", "genre_and_votes"]],
        on="item_id_enc",
        how="left"
    )

    return recommendations

In [50]:
# Выбираем 3 популярные книги
selected_book_ids = (
    events_train.groupby("book_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(3)
    .index
)

for book_id in selected_book_ids:
    original = items.loc[
        items["book_id"] == book_id,
        ["author", "title", "genre_and_votes"]
    ]

    print("\n" + "=" * 70)
    print("ИСХОДНАЯ КНИГА")
    display(original)

    print("ПОХОЖИЕ КНИГИ")
    recommendations = get_similar_books(
        book_id,
        als_model,
        items,
        item_encoder,
        n=5
    )

    display(
        recommendations[
            ["author", "title", "similarity", "genre_and_votes"]
        ]
    )


ИСХОДНАЯ КНИГА


,author,title,genre_and_votes
1967688.0,Paula Hawkins,The Girl on the Train,"{'Fiction': 9793, 'Mystery': 9190, 'Thriller':..."


ПОХОЖИЕ КНИГИ


,author,title,similarity,genre_and_votes
0,Paul Twitchell,Stranger by the River,0.757998,"{'Religion': 1, 'Religion-Christianity': 1, 'R..."
1,Joel Ponzio,F Wall Street: Joe Ponzio's No-Nonsense Approa...,0.753357,"{'Economics-Finance': 11, 'Business': 4, 'Nonf..."
2,Skylar Nightingale,Forbidden Acts,0.753142,None
3,أبو حيان التوحيدي,الإمتاع والمؤانسة / القسم الأول,0.750357,"{'Philosophy': 7, 'Nonfiction': 7}"
4,Satish Batakala,Next Spin : Tryst with greed,0.729359,None



ИСХОДНАЯ КНИГА


,author,title,genre_and_votes
1974280.0,John Green,The Fault in Our Stars,"{'Young Adult': 20449, 'Fiction': 10338, 'Roma..."


... (2 outputs left out)


ИСХОДНАЯ КНИГА


,author,title,genre_and_votes
2208722.0,Veronica Roth,"Insurgent (Divergent, #2)","{'Young Adult': 13994, 'Science Fiction-Dystop..."


ПОХОЖИЕ КНИГИ


,author,title,similarity,genre_and_votes
0,Veronica Roth,"Allegiant (Divergent, #3)",0.972793,"{'Young Adult': 10664, 'Science Fiction-Dystop..."
1,Veronica Roth,"Divergent (Divergent, #1)",0.971671,"{'Young Adult': 20260, 'Science Fiction-Dystop..."
2,M.F.C. Assoum,Strike of Lightning,0.795033,"{'Thriller': 1, 'Fantasy': 1, 'Young Adult': 1..."
3,Renee Altson,Stumbling Toward Faith,0.768060,"{'Spirituality': 1, 'Biography': 1}"
4,Joseph O'Connor,NLP Workbook: A practical guide to achieving t...,0.751913,"{'Psychology': 9, 'Nonfiction': 5, 'Self Help'..."


# === Базовые подходы: валидация

In [51]:
recommendations = recommendations.rename(
    columns={"similarity": "score"}
)
recommendations["user_id"] = user_id

In [52]:
# Пользователи из тестовой выборки
test_users = events_test["user_id"].drop_duplicates()

# Популярные книги, рассчитанные на обучающих данных
popular_books = (
    top_k_pop_items[["book_id", "popularity_score"]]
    .drop_duplicates("book_id")
    .sort_values("popularity_score", ascending=False)
    .head(10)
)

# Формируем рекомендации для тестовых пользователей
recommendations = (
    pd.MultiIndex.from_product(
        [test_users, popular_books["book_id"]],
        names=["user_id", "book_id"]
    )
    .to_frame(index=False)
    .merge(
        popular_books,
        on="book_id",
        how="left"
    )
    .rename(columns={"popularity_score": "score"})
)

print("Пользователей:", recommendations["user_id"].nunique())
print("Рекомендаций:", len(recommendations))

display(recommendations.head())


Пользователей: 138148
Рекомендаций: 1381480


,user_id,book_id,score
0,d986f354a045ffb91234e4af4d1b12fd,18007564,0.397052
1,d986f354a045ffb91234e4af4d1b12fd,16096824,0.326464
2,d986f354a045ffb91234e4af4d1b12fd,3,0.324709
3,d986f354a045ffb91234e4af4d1b12fd,15881,0.273431
4,d986f354a045ffb91234e4af4d1b12fd,17927395,0.260846


In [53]:
def process_events_recs_for_binary_metrics(
    events_train, events_test, recs, top_k=None
):
    """
    Подготовка данных для расчёта Precision, Recall и F1.
    """

    common_users = set(events_test["user_id"]) & set(recs["user_id"])
    print(f"Common users: {len(common_users)}")

    # Реальные взаимодействия
    events_for_common_users = events_test.loc[
        events_test["user_id"].isin(common_users)
        & events_test["book_id"].isin(events_train["book_id"])
        ,
        ["user_id", "book_id"]
    ].drop_duplicates().copy()

    events_for_common_users["gt"] = True

    # Рекомендации модели
    recs_for_common_users = (
        recs.loc[
            recs["user_id"].isin(common_users),
            ["user_id", "book_id", "score"]
        ]
        .sort_values(
            ["user_id", "score"],
            ascending=[True, False]
        )
        .drop_duplicates(["user_id", "book_id"])
    )

    if top_k is not None:
        recs_for_common_users = (
            recs_for_common_users
            .groupby("user_id")
            .head(top_k)
        )

    # Объединяем события и рекомендации
    result = events_for_common_users.merge(
        recs_for_common_users,
        on=["user_id", "book_id"],
        how="outer"
    )

    result["gt"] = result["gt"].fillna(False).astype(bool)
    result["pr"] = result["score"].notna()

    result["tp"] = result["gt"] & result["pr"]
    result["fp"] = ~result["gt"] & result["pr"]
    result["fn"] = result["gt"] & ~result["pr"]

    return result


events_recs_for_binary_metrics = (
    process_events_recs_for_binary_metrics(
        events_train,
        events_test,
        recommendations,
        top_k=5
    )
)

display(events_recs_for_binary_metrics.head())

Common users: 138148


/tmp/ipykernel_1944/3737106105.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["gt"] = result["gt"].fillna(False).astype(bool)


,user_id,book_id,gt,score,pr,tp,fp,fn
0,00009e46d18f223a82b22da38586b605,3,False,0.324709,True,False,True,False
1,00009e46d18f223a82b22da38586b605,15881,False,0.273431,True,False,True,False
2,00009e46d18f223a82b22da38586b605,16096824,False,0.326464,True,False,True,False
3,00009e46d18f223a82b22da38586b605,17927395,False,0.260846,True,False,True,False
4,00009e46d18f223a82b22da38586b605,18007564,False,0.397052,True,False,True,False


In [54]:
def compute_cls_metrics(events_recs_for_binary_metric):

    groupper = events_recs_for_binary_metric.groupby("user_id")

    # Precision = TP / (TP + FP)
    precision = groupper["tp"].sum() / (
        groupper["tp"].sum() + groupper["fp"].sum()
    )
    precision = precision.fillna(0).mean()

    # Recall = TP / (TP + FN)
    recall = groupper["tp"].sum() / (
        groupper["tp"].sum() + groupper["fn"].sum()
    )
    recall = recall.fillna(0).mean()

    return round(precision, 3), round(recall, 3)


precision, recall = compute_cls_metrics(events_recs_for_binary_metrics)

print(f"Precision@5: {precision:.3f}")
print(f"Recall@5: {recall:.3f}")

Precision@5: 0.007
Recall@5: 0.010


In [ ]:
# Метрики для TOP-5
events_recs_at_5 = process_events_recs_for_binary_metrics(
    events_train,
    events_test,
    recommendations,
    top_k=5
)

precision_5, recall_5 = compute_cls_metrics(events_recs_at_5)

# Метрики для TOP-10
events_recs_at_10 = process_events_recs_for_binary_metrics(
    events_train,
    events_test,
    recommendations,
    top_k=10
)

precision_10, recall_10 = compute_cls_metrics(events_recs_at_10)

# Сравнение
print(f"{'Метрика':<15} {'TOP-5':<12} {'TOP-10':<12}")
print("-" * 39)
print(f"{'Precision':<15} {precision_5:<12.3f} {precision_10:<12.3f}")
print(f"{'Recall':<15} {recall_5:<12.3f} {recall_10:<12.3f}")

Common users: 138148


/tmp/ipykernel_1944/3737106105.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["gt"] = result["gt"].fillna(False).astype(bool)


Common users: 138148


/tmp/ipykernel_1944/3737106105.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["gt"] = result["gt"].fillna(False).astype(bool)


Метрика         TOP-5        TOP-10      
---------------------------------------
Precision       0.007        0.007       
Recall          0.010        0.021       


: 

# === Двухстадийный подход: метрики

# === Двухстадийный подход: модель

# === Двухстадийный подход: построение признаков